In [2]:
import os
for f in sorted(os.listdir('data/processed')):
    print(f"{f:32s} {os.path.getsize('data/processed/'+f)/1e6:7.1f} MB")

feature_set.json                     0.0 MB
gains_table.csv                      0.0 MB
scorecard_coefficients.csv           0.0 MB
scorecard_points.csv                 0.0 MB
survival_sample.parquet              6.2 MB
test_binned.parquet                 13.0 MB
test_raw.parquet                    20.5 MB
test_scored.parquet                  8.3 MB
test_scored_xgb.parquet              3.8 MB
train_binned.parquet                15.6 MB
train_raw.parquet                   24.6 MB
walkforward.csv                      0.0 MB
xgb_model.json                       1.7 MB


In [3]:
# ============================================================
# BLOCK A — load from parquet. No CSV, no memory spike.
# ============================================================
import os, gc
os.environ['TMP'] = os.environ['TEMP'] = r'X:\Projects\credit-pd-engine\tmp'
os.makedirs(os.environ['TMP'], exist_ok=True)

import pandas as pd, numpy as np, json
import xgboost as xgb
from sklearn.metrics import roc_auc_score

train_x = pd.read_parquet('data/processed/train_raw.parquet')
test_x  = pd.read_parquet('data/processed/test_raw.parquet')

print(f"Train: {len(train_x):,} | bad rate = {train_x['target'].mean():.3f}")
print(f"Test:  {len(test_x):,} | bad rate = {test_x['target'].mean():.3f}")
assert len(train_x) == 453_780 and len(test_x) == 375_546

Train: 453,780 | bad rate = 0.170
Test:  375,546 | bad rate = 0.202


In [4]:
# ============================================================
# BLOCK B — feature prep
# XGBoost gets RAW values: no binning, no WOE. NaN stays NaN —
# trees learn a default split direction, so missingness is a feature.
# ============================================================
for d in (train_x, test_x):
    ecl = pd.to_datetime(d['earliest_cr_line'], format='%b-%Y', errors='coerce')
    d['cr_hist_yrs']    = (d['issue_d'] - ecl).dt.days / 365.25
    d['term_months']    = d['term'].str.extract(r'(\d+)').astype(float)
    d['emp_length_num'] = (d['emp_length']
        .replace({'< 1 year': '0', '10+ years': '10'})
        .str.extract(r'(\d+)').astype(float))

CAT = ['home_ownership', 'verification_status', 'purpose', 'addr_state']
NUM = ['loan_amnt', 'installment', 'annual_inc', 'dti', 'fico_range_low',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'pub_rec_bankruptcies',
       'cr_hist_yrs', 'term_months', 'emp_length_num']

# Categories frozen on TRAIN — same rule as bin edges in nb 02
for c in CAT:
    cats = pd.Categorical(train_x[c]).categories
    train_x[c] = pd.Categorical(train_x[c], categories=cats)
    test_x[c]  = pd.Categorical(test_x[c],  categories=cats)

XCOLS = NUM + CAT
X_tr, y_tr = train_x[XCOLS], train_x['target']
X_te, y_te = test_x[XCOLS],  test_x['target']

print(f"{len(XCOLS)} features ({len(NUM)} numeric, {len(CAT)} categorical) "
      f"vs 10 in the scorecard")

20 features (16 numeric, 4 categorical) vs 10 in the scorecard


In [5]:
# ============================================================
# BLOCK C — train. Early stopping validates on 2014, NOT 2015.
# Tuning against the test set is the leak the time split exists to prevent.
# ============================================================
inner_tr = train_x['vintage'] <= 2013
inner_va = train_x['vintage'] == 2014
print(f"Inner train (2007-13): {inner_tr.sum():,} | Inner val (2014): {inner_va.sum():,}")

params = dict(
    objective='binary:logistic', eval_metric='auc',
    max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=50,     # large leaves — 450k rows, don't memorise noise
    reg_lambda=1.0,
    enable_categorical=True, tree_method='hist',
    max_bin=128,             # halves histogram memory vs default 256
    n_estimators=1500, early_stopping_rounds=50,
    random_state=42, n_jobs=4,
)

clf = xgb.XGBClassifier(**params)
clf.fit(X_tr[inner_tr], y_tr[inner_tr],
        eval_set=[(X_tr[inner_va], y_tr[inner_va])], verbose=100)

print(f"\nBest iteration: {clf.best_iteration} | Best val AUC: {clf.best_score:.4f}")
gc.collect()

Inner train (2007-13): 230,677 | Inner val (2014): 223,103
[0]	validation_0-auc:0.65956
[100]	validation_0-auc:0.69911
[200]	validation_0-auc:0.70211
[300]	validation_0-auc:0.70294
[400]	validation_0-auc:0.70314
[500]	validation_0-auc:0.70319
[514]	validation_0-auc:0.70314

Best iteration: 464 | Best val AUC: 0.7033


69

In [6]:
# ============================================================
# BLOCK D — evaluate against the scorecard on identical rows
# ============================================================
p_tr = clf.predict_proba(X_tr)[:, 1]
p_te = clf.predict_proba(X_te)[:, 1]

auc_tr_x = roc_auc_score(y_tr, p_tr)
auc_te_x = roc_auc_score(y_te, p_te)

d_ = pd.DataFrame({'y': y_te.values, 'p': p_te}).sort_values('p')
ks_x = ((d_['y'] == 0).cumsum() / (d_['y'] == 0).sum()
        - (d_['y'] == 1).cumsum() / (d_['y'] == 1).sum()).abs().max()

SCORECARD_TEST_AUC, BENCHMARK_AUC = 0.7076, 0.7100

print(f"XGBoost    train AUC : {auc_tr_x:.4f}")
print(f"XGBoost    test  AUC : {auc_te_x:.4f}   (out-of-time, 2015)")
print(f"Scorecard  test  AUC : {SCORECARD_TEST_AUC:.4f}")
print(f"LC sub_grade     AUC : {BENCHMARK_AUC:.4f}")
print(f"\nXGB Gini {2*auc_te_x-1:.4f} | XGB KS {ks_x:.4f}")
print(f"Uplift over scorecard: {auc_te_x - SCORECARD_TEST_AUC:+.4f}")

XGBoost    train AUC : 0.7180
XGBoost    test  AUC : 0.7165   (out-of-time, 2015)
Scorecard  test  AUC : 0.7076
LC sub_grade     AUC : 0.7100

XGB Gini 0.4330 | XGB KS 0.3147
Uplift over scorecard: +0.0089


In [7]:
# ============================================================
# BLOCK E — does XGBoost calibrate any better?
# Scorecard: predicted 0.1720 vs actual 0.2019, Platt slope 1.1671
# ============================================================
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

pt, pp = calibration_curve(y_te, p_te, n_bins=10, strategy='quantile')
cal_x = pd.DataFrame({'predicted_PD': pp.round(4), 'observed_PD': pt.round(4)})
cal_x['gap'] = (cal_x['observed_PD'] - cal_x['predicted_PD']).round(4)
print(cal_x.to_string(index=False))

lg = np.log(np.clip(p_te, 1e-6, 1-1e-6) / (1 - np.clip(p_te, 1e-6, 1-1e-6)))
slope = LogisticRegression().fit(lg.reshape(-1, 1), y_te).coef_[0][0]

print(f"\nXGB mean predicted PD : {p_te.mean():.4f}")
print(f"Actual bad rate       : {y_te.mean():.4f}")
print(f"XGB Platt slope       : {slope:.4f}   (scorecard: 1.1671)")

 predicted_PD  observed_PD    gap
       0.0407       0.0428 0.0021
       0.0682       0.0811 0.0129
       0.0879       0.1066 0.0187
       0.1071       0.1303 0.0232
       0.1277       0.1583 0.0306
       0.1507       0.1862 0.0355
       0.1780       0.2213 0.0433
       0.2129       0.2701 0.0572
       0.2632       0.3389 0.0757
       0.3668       0.4828 0.1160

XGB mean predicted PD : 0.1603
Actual bad rate       : 0.2019
XGB Platt slope       : 1.1470   (scorecard: 1.1671)


In [8]:
# ============================================================
# BLOCK F — SHAP: what did the tree find that the scorecard couldn't?
# ============================================================
import shap

samp = X_te.sample(20_000, random_state=42)
sv = shap.TreeExplainer(clf).shap_values(samp)

imp = pd.Series(np.abs(sv).mean(0), index=XCOLS).sort_values(ascending=False)
print("Mean |SHAP| — top 15:")
print(imp.head(15).round(4).to_string())

in_scorecard = {'term_months', 'fico_range_low', 'dti', 'annual_inc',
                'loan_amnt', 'inq_last_6mths', 'purpose', 'home_ownership',
                'verification_status', 'emp_length_num'}
print("\nTop features NOT in the scorecard:")
print(imp[[f for f in imp.index if f not in in_scorecard]].head(6).round(4).to_string())

Mean |SHAP| — top 15:
term_months       0.3562
annual_inc        0.2616
fico_range_low    0.2468
installment       0.1940
inq_last_6mths    0.1314
open_acc          0.0960
addr_state        0.0945
purpose           0.0896
dti               0.0846
revol_util        0.0839
revol_bal         0.0741
home_ownership    0.0634
total_acc         0.0550
emp_length_num    0.0460
cr_hist_yrs       0.0448

Top features NOT in the scorecard:
installment    0.1940
open_acc       0.0960
addr_state     0.0945
revol_util     0.0839
revol_bal      0.0741
total_acc      0.0550


In [9]:
# ============================================================
# BLOCK G — final comparison, save
# ============================================================
comp = pd.DataFrame({
    'WOE Scorecard': [10, 0.6848, SCORECARD_TEST_AUC, 2*SCORECARD_TEST_AUC-1,
                      0.2986, 0.1720, 1.1671, 'Yes — points table'],
    'XGBoost':       [len(XCOLS), round(auc_tr_x,4), round(auc_te_x,4),
                      round(2*auc_te_x-1,4), round(ks_x,4), round(p_te.mean(),4),
                      round(slope,4), 'No — SHAP only'],
    'LC sub_grade':  ['—', '—', BENCHMARK_AUC, round(2*BENCHMARK_AUC-1,4),
                      '—', '—', '—', 'Proprietary'],
}, index=['Features','Train AUC','Test AUC','Test Gini','Test KS',
          'Mean pred PD','Platt slope','Deployable by hand'])
print(comp.to_string())

test_x[['id']].assign(pd_xgb=p_te, target=y_te.values).to_parquet(
    'data/processed/test_scored_xgb.parquet', index=False)
clf.save_model('data/processed/xgb_model.json')

                         WOE Scorecard         XGBoost LC sub_grade
Features                            10              20            —
Train AUC                       0.6848           0.718            —
Test AUC                        0.7076          0.7165         0.71
Test Gini                       0.4152           0.433         0.42
Test KS                         0.2986          0.3147            —
Mean pred PD                     0.172          0.1603            —
Platt slope                     1.1671           1.147            —
Deployable by hand  Yes — points table  No — SHAP only  Proprietary


In [10]:
# ============================================================
# BLOCK H — is the uplift real, or is it int_rate leaking in?
# installment = f(loan_amnt, term, int_rate). With loan_amnt and
# term already in the frame, the tree can invert to int_rate —
# which is LC's own price, i.e. sub_grade in disguise.
# ============================================================
XCOLS_NL = [c for c in XCOLS if c != 'installment']

clf_nl = xgb.XGBClassifier(**params)
clf_nl.fit(X_tr.loc[inner_tr, XCOLS_NL], y_tr[inner_tr],
           eval_set=[(X_tr.loc[inner_va, XCOLS_NL], y_tr[inner_va])], verbose=False)

auc_nl = roc_auc_score(y_te, clf_nl.predict_proba(X_te[XCOLS_NL])[:, 1])

print(f"XGB with installment    : {auc_te_x:.4f}")
print(f"XGB without installment : {auc_nl:.4f}")
print(f"Cost of removing it     : {auc_te_x - auc_nl:+.4f}")
print(f"Clean uplift over scorecard: {auc_nl - SCORECARD_TEST_AUC:+.4f} "
      f"(was {auc_te_x - SCORECARD_TEST_AUC:+.4f})")

XGB with installment    : 0.7165
XGB without installment : 0.7149
Cost of removing it     : +0.0016
Clean uplift over scorecard: +0.0073 (was +0.0089)
